# Notebook Crash-1: Architecture & Concepts — From Simple to Advanced

---

| Notebook | Topic |
|---|---|
| NB Crash-0 | Data Preprocessing — d3plot → Zarr |
| **NB Crash-1** | **Architecture & Concepts (this notebook)** |
| NB Crash-2 | Training & Integration Methods — one-shot vs time-conditional vs autoregressive |

**Prerequisites:** None. This notebook is pure NumPy — no dataset, no GPU, no prior notebooks required.
Everything about Physics-Attention and GALE is explained from scratch here.

> Run this notebook first if you want the concepts before the code.

### What you will learn
1. Why crash simulation is a hard ML problem
2. The mathematical structure of the prediction task
3. How GeoTransolver applies to structural mechanics
4. The four rollout/integration strategies with NumPy blueprints
5. Why integration strategy matters — qualitative analysis

## Series Overview

```mermaid
flowchart LR
    SG[<b>Notebook Crash-Sim</b><br/>OpenRadioss SimGen<br/><i>generate d3plot dataset</i>]
    C0[<b>Notebook Crash-0</b><br/>Data Preprocessing<br/><i>d3plot to Zarr ETL</i>]
    C1[<b>Notebook Crash-1</b><br/>Architecture and Concepts<br/><i>GeoTransolver theory</i>]
    C2[<b>Notebook Crash-2</b><br/>Training and Comparison<br/><i>train and evaluate</i>]

    SG -->|"d3plot files"| C0
    C0 -->|"Zarr stores"| C2
    C1 -.->|"concepts"| C2

    style C1 fill:#1a6b3a,color:#fff,stroke:#0d4a28,stroke-width:3px
```

> **You are here:** Notebook Crash-1 — the conceptual foundation. No training runs here;
> this notebook builds intuition with NumPy blueprints before Crash-2 does the real training.


## Section 0 — Setup and Imports


In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ.setdefault('WARP_SHOW_DEPRECATION_WARNINGS', '0')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Markdown as md, display
from tabulate import tabulate

print('Environment ready.')

## Section 1 — Why Crash Simulation is a Hard ML Problem

### 1.1 The prediction target

A crash simulation produces a **node trajectory tensor**:

```
X ∈ ℝ^{N × T × 3}
```

where:
- N ≈ 5,000–20,000 nodes (after wall filtering)
- T = 51 timesteps (0–250 ms sampled every 5 ms)
- 3 = x, y, z coordinates

From Notebook Crash-0 we know a single run produces ~117 MB of raw data. The ML model
must learn to predict X from just three scalar conditions (velocity, thickness, wall offset)
and the reference geometry at t=0.

### 1.2 What makes this harder than CFD?

| Property | CFD (Ahmed body, NB3) | Crash (Bumper beam) |
|---|---|---|
| Output type | Steady-state field | **Time-dependent trajectory** |
| Output dim | (N, 4) pressure + WSS | **(N, T, 3) positions** |
| Physics type | Elliptic (Poisson) | **Hyperbolic (wave-like)** |
| Time coupling | None (steady state) | **Autoregressive** |
| Error accumulation | N/A | **Compounds over T steps** |

The temporal dimension T creates a new challenge: **how** do we roll out T=51 predictions?
This is the central design decision in the crash recipe.


In [ ]:
# ── Toy illustration of the prediction task ────────────────────────────────────
# Simulate a simplified 1D crash trajectory to build intuition
np.random.seed(42)
T = 51
dt = 5.0   # ms
t  = np.arange(T) * dt

# Simplified impact model: exponential rise then plateau
def crash_trajectory(v0, k):
    """Simplified 1D bumper displacement: d(t) = v0/k * (1 - exp(-k*t/1000))"""
    return (v0 / k) * (1 - np.exp(-k * t / 1000.0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: different impact velocities
for v0 in [3000, 5000, 7000]:
    axes[0].plot(t, crash_trajectory(v0, 20), label=f'v₀={v0} mm/s')
axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Displacement (mm)')
axes[0].set_title('Effect of Impact Velocity'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: different thickness scales
for ts in [0.7, 1.0, 1.3]:
    axes[1].plot(t, crash_trajectory(5000, 20*ts), label=f'thickness×{ts}')
axes[1].set_xlabel('Time (ms)'); axes[1].set_ylabel('Displacement (mm)')
axes[1].set_title('Effect of Thickness Scale'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Crash Trajectory Sensitivity to Design Parameters', fontweight='bold')
plt.tight_layout(); plt.show()
print('Key insight: the full trajectory shape changes non-linearly with design parameters.')

## Section 2 — GeoTransolver for Structural Mechanics

GeoTransolver was introduced in the Transolver series for CFD. The same architecture applies
to structural crash — the only change is the **input/output format** and the **rollout strategy**.

### 2.1 Architecture recap

```
Inputs at t=0:
  coords     (N, 3)    — reference node positions
  thickness  (N, 1)    — per-node shell thickness
  global_cond (3,)     — [velocity_x, thickness_scale, rwall_origin_y]
  geometry   (M, 3)    — GALE geometry context (downsampled reference mesh)
       │
       ▼
┌─────────────────────────────────────────────────────────────────┐
│                      GeoTransolver                              │
│                                                                 │
│  ┌─────────────┐    ┌─────────────────────────────────┐         │
│  │ GALE Cross- │ ←─ │ Geometry Context  (M, d_model)  │         │
│  │  Attention  │    └─────────────────────────────────┘         │
│  └──────┬──────┘  ↑ repeated at every block                     │
│         │         │                                             │
│  ┌──────▼──────┐  │  ┌─────────────────────────────────┐        │
│  │  Physics-   │  │  │  Transolver blocks × depth      │        │
│  │  Attention  │──┘  │  Slice→Aggregate→Attend→Deslice │        │
│  └──────┬──────┘     └─────────────────────────────────┘        │
└─────────┼───────────────────────────────────────────────────────┘
          │
          ▼
Output:  trajectory (N, T-1, 3) or (N, 3) depending on rollout scheme
```

The **global conditioning vector** (velocity, thickness, wall offset) is injected as an
embedding that shifts and scales every GALE attention block — similar to how diffusion models
inject timestep conditioning.

### 2.2 Data flow through the model

```mermaid
flowchart TD
    subgraph IN ["Inputs at t=0"]
        C[("coords [N,3]<br/>reference node positions")]
        TH[("thickness [N,1]<br/>per-node shell thickness")]
        G[("global_cond [3]<br/>velocity_x, thickness_scale, rwall_origin_y")]
        GEO[("geometry [M,3]<br/>downsampled reference mesh")]
    end

    C --> EMB
    TH --> EMB
    EMB[<b>Input Embedding</b><br/>linear projection to d_model]

    G --> COND
    COND[<b>Conditioning Embedding</b><br/>produces per-block scale and shift<br/>FiLM-style modulation]

    GEO --> GCTX
    GCTX[<b>Geometry Context</b><br/>encoded once, reused every block<br/>M x d_model]

    EMB --> BLK

    subgraph BLOCKS ["Transolver Blocks x depth"]
        BLK[<b>GALE Cross-Attention</b><br/>query = node states<br/>key/value = geometry context<br/>gate alpha controls geometry recall]
        PA[<b>Physics-Attention</b><br/>1. Slice: N nodes to M physics slices<br/>2. Aggregate: weighted token per slice<br/>3. Attend: MxM self-attention<br/>4. Deslice: broadcast back to N nodes]
        FF[<b>Feed-Forward</b><br/>MLP with residual]
        BLK --> PA --> FF
    end

    GCTX -.->|"injected at every block"| BLK
    COND -.->|"modulates every block"| BLK
    COND -.->|"modulates every block"| PA

    FF -->|"repeat x depth"| BLK

    FF --> HEAD

    HEAD[<b>Output Head</b><br/>depends on rollout strategy]

    HEAD --> O1
    HEAD --> O2
    HEAD --> O3

    O1[("One-Shot<br/>[N, T-1, 3]")]
    O2[("Time-Conditional<br/>[N, 3] per tau")]
    O3[("Autoregressive<br/>[N, 3] per step")]

    style EMB fill:#4a7fb5,color:#fff
    style COND fill:#e07b00,color:#fff
    style GCTX fill:#e07b00,color:#fff
    style BLK fill:#6a5acd,color:#fff
    style PA fill:#6a5acd,color:#fff
    style HEAD fill:#1a6b3a,color:#fff
```


In [ ]:
# ── Shape reference table ──────────────────────────────────────────────────────
rows = [
    ['coords (x₀)',  '(N, 3)',       'Reference node positions at t=0'],
    ['thickness',    '(N, 1)',       'Per-node shell thickness from .k file'],
    ['global_cond',  '(3,)',         'velocity_x, thickness_scale, rwall_origin_y'],
    ['geometry',     '(M, 3)',       'Downsampled geometry for GALE cross-attention'],
    ['─── Output (one-shot) ───', '', ''],
    ['trajectory y', '(N, T-1, 3)', 'Predicted node positions at t=1..T-1'],
    ['─── Output (time-conditional) ───', '', ''],
    ['y_t',         '(N, 3)',       'Predicted positions at one timestep t'],
]
print(tabulate(rows, headers=['Tensor', 'Shape', 'Description'], tablefmt='github'))

## Section 3 — The Four Rollout / Integration Strategies

This is the core architectural decision in the crash recipe. The question is:

> **Given the initial state x₀ and global conditions g, how do we produce the full trajectory x₁, x₂, ..., x_{T-1}?**

Four strategies are supported, with very different accuracy / cost trade-offs.

---

### Strategy 1: One-Shot (Direct)

**Analogy:** Like a mathematical function — compute the entire answer at once, no stepping.

```
f(x₀, g) → (x₁, x₂, ..., x_{T-1})    # all T-1 steps in ONE forward pass
```

The model learns to directly map initial conditions to the full trajectory. Think of it as
learning the closed-form solution to the differential equation.

**Pros:** Fast training (one sample = one run), stable (no error accumulation).  
**Cons:** Large output head; may miss fine temporal dynamics.

### The four strategies side by side

```mermaid
flowchart LR
    subgraph S1 ["Strategy 1 · One-Shot"]
        A1["x0, g"] --> F1["f"] --> B1["x1, x2, ... xT-1<br/>all at once"]
    end

    subgraph S2 ["Strategy 2 · Time-Conditional"]
        A2["x0, g, tau=t/(T-1)"] --> F2["f"] --> B2["xt<br/>one step"]
        B2 -.->|"call T-1 times<br/>with different tau"| A2
    end

    subgraph S3 ["Strategy 3 · Autoregressive"]
        A3["xt, g"] --> F3["f"] --> B3["xt+1"]
        B3 -->|"feed back as input<br/>errors compound"| A3
    end

    subgraph S4 ["Strategy 4 · Teacher Forcing"]
        A4["xt GROUND TRUTH, g"] --> F4["f"] --> B4["xt+1"]
        B4 -.->|"train-test mismatch:<br/>inference must use own predictions"| A4
    end

    style F1 fill:#1a6b3a,color:#fff
    style F2 fill:#1a6b3a,color:#fff
    style F3 fill:#e07b00,color:#fff
    style F4 fill:#c0392b,color:#fff
```

> **Colour key:** green = stable in practice · orange = works but accumulates error ·
> red = low training loss but poor generalization.


In [ ]:
# ── NumPy blueprint: One-Shot rollout ─────────────────────────────────────────
class GeoTransolverOneShot:
    """Simplified blueprint — one forward pass predicts ALL timesteps."""

    def __init__(self, N=1000, d_model=64, T=51, M=16):
        self.N, self.d_model, self.T, self.M = N, d_model, T, M
        # Output head: maps hidden state to (T-1) × 3 per node
        self.W_out = np.random.randn(d_model, (T-1)*3) * 0.01

    def forward(self, x0, global_cond):
        """
        x0          : (N, 3)  — reference positions
        global_cond : (3,)    — [velocity_x, thickness_scale, rwall_origin_y]
        Returns y   : (N, T-1, 3)  — predicted trajectory
        """
        N, T, M = self.N, self.T, self.M

        # 1. Encode: map (N,3) + global_cond → hidden state (N, d_model)
        h = np.random.randn(N, self.d_model)  # placeholder for actual encoding

        # 2. Physics-Attention + GALE blocks (simplified)
        h = h + global_cond.mean()  # global conditioning injection (simplified)

        # 3. Output head: (N, d_model) → (N, (T-1)*3) → (N, T-1, 3)
        y_flat = h @ self.W_out                   # (N, (T-1)*3)
        y      = y_flat.reshape(N, T-1, 3)         # (N, T-1, 3)
        return y

# Demo
model_os  = GeoTransolverOneShot(N=500, T=51)
x0        = np.random.randn(500, 3)
g         = np.array([-5000.0, 1.0, 0.0])
trajectory = model_os.forward(x0, g)
print(f'One-shot output shape: {trajectory.shape}  → (N={trajectory.shape[0]}, T-1={trajectory.shape[1]}, 3)')

### Strategy 2: Time-Conditional

**Analogy:** Ask the function 'what is the state at time t?' for each t independently.

```
f(x₀, g, t/(T-1)) → x_t    # predict ONE step, conditioned on normalized time
```

The normalized time `τ = t/(T-1) ∈ [0, 1]` is appended to the global conditioning vector.
At inference, we call the model T-1 times (one per timestep) to build the full trajectory.

**Pros:** Best accuracy — model focuses on each timestep independently, no output compression.  
**Cons:** Training cost = T-1× higher (one sample per timestep per run).


In [ ]:
# ── NumPy blueprint: Time-Conditional rollout ─────────────────────────────────
class GeoTransolverTimeConditional:
    """One forward pass predicts ONE timestep, conditioned on normalized time τ."""

    def __init__(self, N=1000, d_model=64, M=16):
        self.N, self.d_model, self.M = N, d_model, M
        self.W_out = np.random.randn(d_model, 3) * 0.01

    def forward_step(self, x0, global_cond, tau):
        """
        x0          : (N, 3)  — reference positions (always t=0)
        global_cond : (3,)    — design parameters
        tau         : float   — normalized time t/(T-1) ∈ [0, 1]
        Returns x_t : (N, 3)  — predicted positions at time t
        """
        N = self.N
        # Global cond extended with normalized time: [velocity_x, thickness_scale, rwall_origin_y, tau]
        g_extended = np.append(global_cond, tau)   # (4,)

        # Encode + Physics-Attention + GALE (simplified placeholder)
        h   = np.random.randn(N, self.d_model)
        h   = h + g_extended.mean()
        x_t = h @ self.W_out    # (N, 3)
        return x_t

    def rollout(self, x0, global_cond, T=51):
        """Full trajectory at inference: call forward_step T-1 times."""
        trajectory = np.zeros((T-1, self.N, 3))
        for t in range(1, T):
            tau = t / (T - 1)
            trajectory[t-1] = self.forward_step(x0, global_cond, tau)
        return trajectory.transpose(1, 0, 2)   # (N, T-1, 3)

model_tc   = GeoTransolverTimeConditional(N=500)
trajectory = model_tc.rollout(x0, g, T=51)
print(f'Time-conditional rollout shape: {trajectory.shape}  → (N, T-1, 3)')

### Strategy 3: Autoregressive Rollout (Euler-like)

**Analogy:** Classic numerical integration — use the current state to predict the next.

```
x_{t+1} = f(x_t, g)    # step from current state to next
```

This is the ML equivalent of **explicit Euler integration** of the ODE `dx/dt = f(x, g)`.
The model sees the current mesh state at each step and predicts the next position.

**Pros:** Physically principled; model sees realistic intermediate states.  
**Cons:** Error accumulates over T steps (each small error is input to the next step).
In practice this is often **unstable** for T ≥ 30 — see the benchmark table in NB2.

---

### Strategy 4: Teacher Forcing

**Analogy:** Training with the answer sheet — feed ground-truth intermediate states instead of predictions.

```
Training:   x_{t+1} = f(x_t^{GT}, g)   # use ground truth x_t as input
Inference:  x_{t+1} = f(x_t^{pred}, g) # must use own predictions
```

Teacher forcing is a **training trick**, not an independent architecture. The model learns
to make accurate single-step predictions, but at inference it must use its own (imperfect)
predictions as inputs — creating a **train-test distribution mismatch**.

**Result:** Very low training loss, poor generalization. The benchmark table in NB2
shows validation MSE ~0.3 for teacher forcing vs ~0.004 for one-shot on the same data.


In [ ]:
# ── NumPy blueprint: Autoregressive (Euler-like) rollout ──────────────────────
class GeoTransolverAutoregressive:
    """Each step uses the previous predicted state as input — Euler integration analog."""

    def __init__(self, N=1000, d_model=64, M=16):
        self.N, self.d_model, self.M = N, d_model, M
        self.W_out = np.random.randn(d_model, 3) * 0.01

    def forward_step(self, x_current, global_cond):
        """
        x_current   : (N, 3)  — current mesh positions
        global_cond : (3,)    — design parameters
        Returns x_next : (N, 3)  — positions at next timestep
        """
        N = self.N
        h      = np.random.randn(N, self.d_model)   # encode x_current
        h      = h + global_cond.mean()
        dx     = h @ self.W_out                      # predicted displacement Δx
        return x_current + dx                         # Euler step: x_{t+1} = x_t + Δx

    def rollout(self, x0, global_cond, T=51):
        """Autoregressively predict full trajectory."""
        trajectory = []
        x_curr = x0.copy()
        for _ in range(T-1):
            x_next = self.forward_step(x_curr, global_cond)
            trajectory.append(x_next)
            x_curr = x_next   # ← feed prediction back in (error accumulates here!)
        return np.stack(trajectory, axis=1)   # (N, T-1, 3)

model_ar   = GeoTransolverAutoregressive(N=500)
trajectory = model_ar.rollout(x0, g, T=51)
print(f'Autoregressive rollout shape: {trajectory.shape}  → (N, T-1, 3)')

## Section 4 — Method Comparison Table

The table below summarises the trade-offs across all four strategies.


In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
rows = [
    ['One-Shot',         'Direct mapping x₀ → full trajectory',         'Low (1 sample/run)',    'High (stable)',     '5.42e-3'],
    ['Time-Conditional', 'f(x₀, g, τ) → x_t for each t',               'High (T-1 samples/run)','Best accuracy',     '4.12e-3'],
    ['AR-Rollout',       'f(x_t, g) → x_{t+1}, autoregressively',       'Medium',               'Often unstable',    'unstable'],
    ['Teacher-Forcing',  'AR training with GT intermediate states',      'High (T-1 samples/run)','Poor (mismatch)',   '0.3'],
]
print(tabulate(rows,
               headers=['Strategy', 'Mechanism', 'Training Cost', 'Accuracy', 'Val MSE (Bumper T=50)'],
               tablefmt='github'))
print('\nSource: arXiv:2510.15201, Table comparing integration schemes on Bumper dataset.')

## Section 5 — Visual Intuition: Why Error Accumulates in Autoregressive Rollout

The error accumulation problem in AR rollout is analogous to **Euler's method for ODEs**.
A small per-step error ε grows over T steps — and because the model was never trained on
its own errors (only on ground-truth inputs), the errors compound fast.


In [ ]:
# ── Error accumulation demonstration ─────────────────────────────────────────
T   = 51
eps = 0.01   # small per-step error

# True trajectory (simplified damped oscillation)
t_ax       = np.arange(T)
x_true     = 50 * (1 - np.exp(-0.1 * t_ax)) * np.cos(0.3 * t_ax)

# One-shot: uniform small error, no accumulation
x_oneshot  = x_true + np.random.randn(T) * 2.0   # constant noise floor

# Autoregressive: each step adds eps, errors compound
x_ar = np.zeros(T)
x_ar[0] = x_true[0]
for t in range(1, T):
    dx_true = x_true[t] - x_true[t-1]
    dx_err  = np.random.randn() * eps * (1 + 0.1*t)  # growing error
    x_ar[t] = x_ar[t-1] + dx_true + dx_err

# Time-conditional: independent prediction at each t, no accumulation
x_tc = x_true + np.random.randn(T) * 1.5   # similar to one-shot but often smaller

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(t_ax, x_true,    'k-',  lw=2,   label='Ground truth')
ax.plot(t_ax, x_oneshot, 'b--', lw=1.5, label='One-shot', alpha=0.8)
ax.plot(t_ax, x_tc,      'g--', lw=1.5, label='Time-conditional', alpha=0.8)
ax.plot(t_ax, x_ar,      'r-',  lw=1.5, label='Autoregressive (AR)', alpha=0.8)
ax.set_xlabel('Timestep'); ax.set_ylabel('Displacement')
ax.set_title('Trajectory Prediction Comparison'); ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(t_ax, np.abs(x_oneshot - x_true), 'b-',  lw=1.5, label='One-shot error')
ax2.plot(t_ax, np.abs(x_tc - x_true),      'g-',  lw=1.5, label='Time-cond error')
ax2.plot(t_ax, np.abs(x_ar - x_true),      'r-',  lw=1.5, label='AR error')
ax2.set_xlabel('Timestep'); ax2.set_ylabel('|Error|')
ax2.set_title('Error vs Timestep — AR error grows, others stay bounded')
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_yscale('log')

plt.suptitle('Rollout Strategy Comparison: Error Accumulation', fontweight='bold')
plt.tight_layout(); plt.show()

print('Key insight: AR-rollout errors compound exponentially.')
print('One-shot and time-conditional remain bounded because they always condition on x₀.')

## Section 6 — Physics-Attention in the Crash Context

Crash meshes have 5,000–20,000 nodes. Standard self-attention scales as O(N²) — at N=10,000
this means 10⁸ operations per attention step, infeasible without memory optimizations.

Physics-Attention from Transolver (NB2) reduces this to **O(N)** using M=16–32 physics slices:

```
Standard self-attention:   O(N²)         at N=10,000 → 10⁸ ops
Physics-Attention (M=16):  O(N·M + M²)   at N=10,000, M=16 → 1.6×10⁵ ops  (625× reduction)
```

For crash specifically, the M slices capture **structural zones**:

- Slice 0: front face (primary impact zone, large deformation)
- Slice 1: flanges (moderate bending)
- Slice 2: rear face (small deformation)
- Slice 3–M: transition and boundary zones

After sufficient training, visualizing the ascription matrix A ∈ ℝ^{N×M} should reveal
these physically meaningful groupings — exactly as shown in Notebook Crash-2's visualization section.


In [ ]:
# ── Complexity comparison ──────────────────────────────────────────────────────
N_values = [1_000, 5_000, 10_000, 20_000, 50_000]
M        = 16

rows = []
for N in N_values:
    std_attn  = N**2
    phys_attn = N*M + M**2
    ratio     = std_attn / phys_attn
    rows.append([f'{N:,}', f'{std_attn:,.0f}', f'{phys_attn:,.0f}', f'{ratio:.0f}×'])

print(tabulate(rows,
               headers=['Nodes N', 'Std attention O(N²)', f'Physics-attn O(NM+M²) M={M}', 'Speedup'],
               tablefmt='github'))

fig, ax = plt.subplots(figsize=(9, 4))
ns = np.array(N_values)
ax.loglog(ns, ns**2, 'b-o', lw=2, label='Standard self-attention O(N²)')
ax.loglog(ns, ns*M + M**2, 'r-s', lw=2, label=f'Physics-Attention O(NM+M²), M={M}')
ax.set_xlabel('Mesh nodes N'); ax.set_ylabel('Operations')
ax.set_title('Attention Complexity: Crash Mesh Size Range')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

## Section 7 — GALE in the Crash Context

In CFD (Transolver series NB4), GALE injected **surface geometry** (Ahmed body shape) into
every attention layer. For crash, GALE injects **reference mesh geometry** (bumper beam shape)
to ensure the model always knows the exact structural topology it is simulating.

```
Why GALE matters for crash:

Without GALE:                        With GALE:
  Layer 1: sees x₀ geometry            Every layer sees geometry context C
  Layer 2: hidden state drifts          C is the reference mesh at t=0
  Layer 3: geometry info diluted        α gate controls how much geometry
  ...                                   vs. physics evolution to trust
  Output: may predict a mesh            Output: geometry-consistent trajectory
  inconsistent with the actual part     throughout all T timesteps
```

The learned gate α tends to be higher (more geometry recall) near complex structural
features (welds, flanges, holes) where the deformation mode is strongly shape-dependent.


In [ ]:
# ── GALE gate α illustration ───────────────────────────────────────────────────
# Show how α might vary across the bumper beam surface
np.random.seed(0)
N = 1000
# Simulate node positions along a bumper beam (simplified 1D structure)
x_pos = np.linspace(-600, 600, N)   # x-coordinate (mm)
y_pos = np.random.randn(N) * 20     # y-perturbation

# α tends to be higher at structural features (edges, welds)
# Simplified: higher at the ends (flanges) and center (impact zone)
alpha = 0.3 + 0.4 * (np.sin(np.pi * np.abs(x_pos) / 600)**2) + 0.15 * np.exp(-(x_pos/60)**2)
alpha = np.clip(alpha + np.random.randn(N)*0.05, 0, 1)

fig, ax = plt.subplots(figsize=(12, 4))
sc = ax.scatter(x_pos, y_pos, c=alpha, cmap='RdYlGn_r', s=8, vmin=0, vmax=1)
plt.colorbar(sc, label='GALE gate α  (1=geometry-dominated, 0=physics-dominated)')
ax.set_xlabel('Bumper beam X-position (mm)'); ax.set_ylabel('Y-position (mm)')
ax.set_title('Illustrative GALE gate α across bumper beam nodes\n(red = trust geometry, green = trust physics evolution)')
ax.axvline(-500, color='gray', ls='--', alpha=0.5, label='Flange region')
ax.axvline(+500, color='gray', ls='--', alpha=0.5)
ax.axvline(0, color='navy', ls='--', alpha=0.5, label='Impact center')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print('Note: actual α values emerge from training — this is illustrative.')

## Summary

In this notebook you learned:

1. **The crash prediction task** is harder than CFD: we predict a temporal trajectory
   X ∈ ℝ^{N×T×3} rather than a steady-state field.

2. **GeoTransolver** applies directly to structural mechanics — the GALE geometry conditioning
   ensures each attention layer stays grounded in the reference mesh geometry.

3. **Four rollout strategies** with different accuracy/cost trade-offs:
   - **One-shot**: fast, stable, competitive accuracy — best for fast iteration
   - **Time-conditional**: best accuracy, T× training cost — preferred for deployment
   - **AR-rollout**: often unstable for T ≥ 30 due to error accumulation (Euler analogy)
   - **Teacher-forcing**: low training loss but train-test mismatch → poor generalization

4. **Physics-Attention** reduces O(N²) to O(NM) — crucial at N=5,000–20,000 crash mesh nodes.

5. **GALE** prevents geometry drift across deep networks — especially important at complex
   structural features like flanges and welds.

### What's next

**Notebook Crash-2** puts everything together:

- Full PhysicsNeMo training pipeline on the BumperBeam Zarr data
- Training and comparing one-shot vs time-conditional vs AR-rollout
- Inference, L² error analysis, and deformation visualization

---

**References**

- [arXiv:2510.15201 — Automotive Crash Dynamics Modeling](https://arxiv.org/abs/2510.15201)
- [GeoTransolver paper (arXiv:2512.20399)](https://arxiv.org/abs/2512.20399)
- [Transolver paper (arXiv:2402.02366)](https://arxiv.org/abs/2402.02366)
- [PhysicsNeMo Crash Recipe](https://github.com/NVIDIA/physicsnemo/tree/main/examples/structural_mechanics/crash)
